# Wikipedia / Wikimedia verification

**Source:** Wikipedia / Wikimedia (English)

**Packages:** mwviews (pageviews via the Wikimedia REST API); requests (MediaWiki Action API for title resolution and article text)

**Purpose:** Verify the core capabilities the public profile axis depends on:

*    Naming: can a UFCStats fighter name be mapped to its exact Wikipedia article title?
*    Public profile axis data: are daily article pageviews available via mwviews, in baseline and fight-window forms?
*    Article text: can clean article text be pulled? This was explored as input to the sentence-transformer embeddings (all-MiniLM-L6-v2); that strand was later descoped as a negative result, so this checks the text pull itself rather than a signal that reached the final axis.

**Sample:** the same 10 fighters used in the thin slice, so Wikipedia coverage can be compared against the other signals then under consideration (a small coverage check; Google Trends was later dropped and the final axis pairs Wikipedia pageviews with GDELT).

## Section 1: Setup and imports


In [ ]:
# BLOCK 1: Install and import

!pip install mwviews -q

import requests
import pandas as pd
import time
from datetime import datetime
from mwviews.api import PageviewsClient

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Required due to Wikimedia policy
USER_AGENT = "UFC-Value-Mapper/0.1 (MSc academic project; contact via GitHub th1555)"

print(f"Run date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

  Preparing metadata (setup.py) ... done
Run date: 2026-05-25 01:07:46


In [ ]:
# BLOCK 2: Sample fighters

# Strawweight Comparison Group

sample_fighters = [
    'Rose Namajunas',
    'Ashley Yoder',
    'Gloria de Paula',
    'Tatiana Suarez',
    'Shauna Bannon',
    'Jessica Penne',
    'Amanda Ribas',
    'Alexia Thainara',
    'Denise Gomes',
    'Yan Xiaonan',
]

print(f"Sample fighters: {len(sample_fighters)}")
for f in sample_fighters:
    print(f"  - {f}")

Sample fighters: 10
  - Rose Namajunas
  - Ashley Yoder
  - Gloria de Paula
  - Tatiana Suarez
  - Shauna Bannon
  - Jessica Penne
  - Amanda Ribas
  - Alexia Thainara
  - Denise Gomes
  - Yan Xiaonan


## Section 2: Mapping and Integration

A UFCStats fighter name is not guaranteed to be a Wikipedia article title. Wikipedia uses exact titles with disambiguation conventions (e.g. `Michael Page (fighter)` rather than `Michael Page`). Unresolved names return nothing, no errors are given.

MediaWiki `opensearch` endpoints are used to resolve each fighter name to its best-matching article title, then records the hit rate and failure modes. This is the most important thing the verification establishes; downstream capabilities depend on a correct title!

In [ ]:
# BLOCK 3: Resolve fighter names to Wikipedia article titles
# The closest matching article titles for a search term are provided and the
# top hit is taken. NOT A VERIFICATION

MEDIAWIKI_API = "https://en.wikipedia.org/w/api.php"

def resolve_title(name, user_agent, timeout=10):
    params = {
        "action": "opensearch",
        "search": name,
        "limit": 3,
        "namespace": 0,
        "format": "json",
    }
    try:
        r = requests.get(
            MEDIAWIKI_API,
            params=params,
            headers={"User-Agent": user_agent},
            timeout=timeout
        )
        r.raise_for_status()
        data = r.json()
        # opensearch returns [search_term, [titles], [descriptions], [urls]]
        titles = data[1] if len(data) > 1 else []
        return titles
    except Exception as e:
        print(f"    resolve error for '{name}': {type(e).__name__}: {e}")
        return []

# Resolve each fighter
resolution_results = []
for i, fighter in enumerate(sample_fighters):
    print(f"[{i+1}/{len(sample_fighters)}] {fighter}")
    candidates = resolve_title(fighter, USER_AGENT)
    top_match = candidates[0] if candidates else None
    print(f"    candidates: {candidates}")
    resolution_results.append({
        'fighter': fighter,
        'resolved_title': top_match,
        'all_candidates': candidates,
        'resolved': top_match is not None,
    })
    time.sleep(0.5)

df_resolution = pd.DataFrame(resolution_results)
n_resolved = df_resolution['resolved'].sum()
print()
print(f"Resolution hit rate: {n_resolved}/{len(sample_fighters)} ({100*n_resolved/len(sample_fighters):.0f}%)")
print()
print(df_resolution[['fighter', 'resolved_title', 'resolved']].to_string(index=False))

[1/10] Rose Namajunas
    candidates: ['Rose Namajunas']
[2/10] Ashley Yoder
    candidates: ['Ashley Yoder', 'Ashley Roberts', 'Ashley Bouder']
[3/10] Gloria de Paula
    candidates: ['Gloria de Paula']
[4/10] Tatiana Suarez
    candidates: ['Tatiana Suarez']
[5/10] Shauna Bannon
    candidates: ['Shane Bannon']
[6/10] Jessica Penne
    candidates: ['Jessica Penne', 'Jessica Ennis-Hill', 'Jessica Bennett (journalist)']
[7/10] Amanda Ribas
    candidates: ['Amanda Ribas', 'Amanda Rishworth', 'Amanda Reason']
[8/10] Alexia Thainara
    candidates: []
[9/10] Denise Gomes
    candidates: ['Denise Gomes', 'Denise Jones Ennett', 'Denise Goldsworthy']
[10/10] Yan Xiaonan
    candidates: ['Yan Xiaonan', 'Yan Xiaoling – Fan Yanqiong case', 'Yao Xiaotang']

Resolution hit rate: 9/10 (90%)

        fighter  resolved_title  resolved
 Rose Namajunas  Rose Namajunas      True
   Ashley Yoder    Ashley Yoder      True
Gloria de Paula Gloria de Paula      True
 Tatiana Suarez  Tatiana Suarez      Tru

## Section 3: Pageviews via mwviews (public profile axis)

Daily pageviews over a set cycle are pulled for all resolved fighters. Confirms the data shape, and the granularity that is available the public profile axis.


In [ ]:
# BLOCK 4: Pull pageviews for resolved fighters
# Daily views per article; this is the public-profile attention signal.

# PageviewsClient with USER_AGENT
pv_client = PageviewsClient(user_agent=USER_AGENT)

# Sample window
PV_START = '20250501'
PV_END = '20260501'

# Only query resolved fighters
resolved_titles = df_resolution[df_resolution['resolved']]['resolved_title'].tolist()
print(f"Pulling pageviews for {len(resolved_titles)} resolved fighters")
print(f"Window: {PV_START} to {PV_END} (daily)")
print()

pageview_summary = []
for title in resolved_titles:
    try:
        # article_views returns
        views = pv_client.article_views(
            'en.wikipedia',
            [title],
            granularity='daily',
            start=PV_START,
            end=PV_END
        )
        # Extract article's daily series
        title_key = title.replace(' ', '_')
        daily_values = [day_data.get(title_key) for day_data in views.values()]
        daily_values = [v for v in daily_values if v is not None]

        if daily_values:
            avg_views = sum(daily_values) / len(daily_values)
            max_views = max(daily_values)
            pageview_summary.append({
                'title': title,
                'n_days': len(daily_values),
                'avg_daily_views': round(avg_views, 1),
                'max_daily_views': max_views,
            })
            print(f"  {title}: avg {avg_views:.0f}/day, peak {max_views}")
        else:
            print(f"  {title}: no pageview data returned")
            pageview_summary.append({
                'title': title, 'n_days': 0,
                'avg_daily_views': None, 'max_daily_views': None,
            })
    except Exception as e:
        print(f"  {title}: FAIL {type(e).__name__}: {e}")
        pageview_summary.append({
            'title': title, 'n_days': 0,
            'avg_daily_views': None, 'max_daily_views': None,
        })
    time.sleep(0.5)

df_pageviews = pd.DataFrame(pageview_summary)
print()
print("Pageview summary:")
print(df_pageviews.to_string(index=False))

Pulling pageviews for 9 resolved fighters
Window: 20250501 to 20260501 (daily)

  Rose Namajunas: avg 1555/day, peak 58804
  Ashley Yoder: avg 35/day, peak 218
  Gloria de Paula: avg 19/day, peak 171
  Tatiana Suarez: avg 470/day, peak 9955
  Shane Bannon: avg 10/day, peak 32
  Jessica Penne: avg 58/day, peak 160
  Amanda Ribas: avg 232/day, peak 6883
  Denise Gomes: avg 70/day, peak 1842
  Yan Xiaonan: avg 170/day, peak 3972

Pageview summary:
          title  n_days  avg_daily_views  max_daily_views
 Rose Namajunas     366           1554.7            58804
   Ashley Yoder     366             35.4              218
Gloria de Paula     366             19.3              171
 Tatiana Suarez     366            469.6             9955
   Shane Bannon     366              9.6               32
  Jessica Penne     366             57.8              160
   Amanda Ribas     366            231.7             6883
   Denise Gomes     366             69.5             1842
    Yan Xiaonan     366      

## Section 4: Article text via MediaWiki API (embeddings)

For the resolved fighters, the article's plain-text extract is pulled. This was intended to feed the sentence-transformer embeddings (all-MiniLM-L6-v2) explored for the fighter-similarity work; that strand was later descoped as a negative result, so this section verifies the text pull itself rather than a signal that reached the final framework.

The check confirms clean plain text is extracted and records the text length distribution, since embedding models have input length limits and very short stub articles carry little narrative signal. It also acts as a check that a resolved title is actually about the fighter, since the intro text should mention MMA or UFC.

In [ ]:
# BLOCK 5: Pull article text for resolved fighters

def get_article_text(title, user_agent, intro_only=True, timeout=10):
    params = {
        "action": "query",
        "prop": "extracts",
        "titles": title,
        "explaintext": 1,
        "format": "json",
        "redirects": 1,        # follow redirects
    }
    if intro_only:
        params["exintro"] = 1
    try:
        r = requests.get(
            MEDIAWIKI_API,
            params=params,
            headers={"User-Agent": user_agent},
            timeout=timeout
        )
        r.raise_for_status()
        pages = r.json()['query']['pages']
        page = next(iter(pages.values()))
        return page.get('extract', '')
    except Exception as e:
        print(f"    text error for '{title}': {type(e).__name__}: {e}")
        return ''

text_summary = []
for title in resolved_titles:
    # Intro for relevance check
    intro = get_article_text(title, USER_AGENT, intro_only=True)
    # Full text for length check
    full = get_article_text(title, USER_AGENT, intro_only=False)

    # Relevance check
    intro_lower = intro.lower()
    looks_like_fighter = any(
        kw in intro_lower for kw in ['mixed martial', 'mma', 'ufc', 'fighter', 'fighting']
    )

    text_summary.append({
        'title': title,
        'intro_chars': len(intro),
        'full_chars': len(full),
        'looks_like_fighter': looks_like_fighter,
    })
    flag = 'OK' if looks_like_fighter else 'CHECK (intro does not mention MMA)'
    print(f"  {title}: intro {len(intro)} chars, full {len(full)} chars  [{flag}]")
    time.sleep(0.5)

df_text = pd.DataFrame(text_summary)
print()
print("Text summary:")
print(df_text.to_string(index=False))

  Rose Namajunas: intro 406 chars, full 18736 chars  [OK]
  Ashley Yoder: intro 217 chars, full 5194 chars  [OK]
  Gloria de Paula: intro 217 chars, full 2931 chars  [OK]
  Tatiana Suarez: intro 702 chars, full 9084 chars  [OK]
  Shane Bannon: intro 222 chars, full 1235 chars  [CHECK (intro does not mention MMA)]
  Jessica Penne: intro 316 chars, full 10216 chars  [OK]
  Amanda Ribas: intro 301 chars, full 5771 chars  [OK]
  Denise Gomes: intro 257 chars, full 2840 chars  [OK]
  Yan Xiaonan: intro 341 chars, full 4509 chars  [OK]

Text summary:
          title  intro_chars  full_chars  looks_like_fighter
 Rose Namajunas          406       18736                True
   Ashley Yoder          217        5194                True
Gloria de Paula          217        2931                True
 Tatiana Suarez          702        9084                True
   Shane Bannon          222        1235               False
  Jessica Penne          316       10216                True
   Amanda Ribas       

## Section 5: Coverage comparison

This is an early coverage check across the signals then under consideration: Google Trends, Wikipedia pageviews, and Wikipedia article text. The final public profile axis dropped Trends (unreliable client, independent per-query normalisation) and descoped the article-text embeddings (they captured biographical rather than competitive similarity), settling on Wikipedia pageviews and GDELT news volume. The comparison is retained because its finding holds regardless of the specific signals: different sources leave different fighters uncovered, which is the argument for a multi-source axis.

In [ ]:
# BLOCK 6: Cross source coverage table

trends_success = {
    'Rose Namajunas': True, 'Ashley Yoder': True, 'Gloria de Paula': True,
    'Tatiana Suarez': True, 'Shauna Bannon': True, 'Jessica Penne': True,
    'Amanda Ribas': True, 'Alexia Thainara': True, 'Denise Gomes': True,
    'Yan Xiaonan': True,
}

# Build comparison
coverage_rows = []
for fighter in sample_fighters:
    res_row = df_resolution[df_resolution['fighter'] == fighter].iloc[0]
    wiki_resolved = res_row['resolved']
    resolved_title = res_row['resolved_title']

    # Did pageviews come back?
    pv_ok = False
    if resolved_title and resolved_title in df_pageviews['title'].values:
        pv_row = df_pageviews[df_pageviews['title'] == resolved_title].iloc[0]
        pv_ok = pv_row['n_days'] > 0

    # Did text come back and look like a fighter?
    text_ok = False
    if resolved_title and resolved_title in df_text['title'].values:
        text_row = df_text[df_text['title'] == resolved_title].iloc[0]
        text_ok = text_row['looks_like_fighter']

    coverage_rows.append({
        'fighter': fighter,
        'trends': trends_success.get(fighter, False),
        'wiki_title_resolved': wiki_resolved,
        'wiki_pageviews': pv_ok,
        'wiki_text_ok': text_ok,
    })

df_coverage = pd.DataFrame(coverage_rows)
print("Cross-source coverage (sample of 10 fighters):")
print(df_coverage.to_string(index=False))
print()

# Count coverage level
df_coverage['n_sources'] = df_coverage[['trends', 'wiki_pageviews', 'wiki_text_ok']].sum(axis=1)
print("Coverage distribution:")
print(f"  All 3 signals: {(df_coverage['n_sources'] == 3).sum()}")
print(f"  1-2 signals: {((df_coverage['n_sources'] >= 1) & (df_coverage['n_sources'] < 3)).sum()}")
print(f"  0 signals: {(df_coverage['n_sources'] == 0).sum()}")

Cross-source coverage (sample of 10 fighters):
        fighter  trends  wiki_title_resolved  wiki_pageviews  wiki_text_ok
 Rose Namajunas    True                 True            True          True
   Ashley Yoder    True                 True            True          True
Gloria de Paula    True                 True            True          True
 Tatiana Suarez    True                 True            True          True
  Shauna Bannon    True                 True            True         False
  Jessica Penne    True                 True            True          True
   Amanda Ribas    True                 True            True          True
Alexia Thainara    True                False           False         False
   Denise Gomes    True                 True            True          True
    Yan Xiaonan    True                 True            True          True

Coverage distribution:
  All 3 signals: 8
  1-2 signals: 2
  0 signals: 0


## Section 6: Analysis

### Verified

**Resolution**
- Raw hit rate: 9 of 10 resolved to a title
- TRUE hit rate: 8 of 10 (Shauna Bannon resolved to "Shane Bannon"; caught by the Section 4 relevance check)

**Pageviews (mwviews)**
- Data: 9 of 9 resolved titles returned full daily series
- Values: avg 10-1,555/day; clear fight-week peaks (Rose Namajunas peak 58,804 vs 1,555 avg)

**Article text (MediaWiki API)**
- Clean plain text returned (explaintext=1 strips markup): confirmed
- Full text length: confirmed
- Relevance check (intro mentions MMA): 8 of 9 resolved titles passed; Shane Bannon failed (correctly, it is the wrong person)

### Cross-source coverage finding

Across the three signals then under consideration (Trends, Wikipedia pageviews, article text):
- 8 of 10 fighters had all three
- 2 of 10 had partial coverage, failing on different sources:
  - Alexia Thainara: a Trends signal was available, but no Wikipedia article exists
  - Shauna Bannon: pageviews resolved, but to the wrong person, so no usable text signal
- 0 of 10 had no coverage at all

The durable finding, independent of which signals are used, is that different sources leave different fighters uncovered, so no single source is complete and a fighter missing from one may be present in another. This is the empirical argument for building the public profile axis on more than one source, which the final design does with Wikipedia pageviews and GDELT.

### Notes

The relevance check is mandatory in the resolver; string matching alone produces silent false positives (Shauna > Shane).

"No article" is a valid state (NaN for the Wikipedia signals), handled the same way as any other missing signal: dropped from that fighter's average rather than counted as a zero.